# 项目：猫狗识别

**1. 项目概述**  
使用深度学习中的卷积神经网络（CNN），对 Kaggle “Dogs vs. Cats” 数据集进行二分类（猫/狗）。项目涵盖数据预处理、模型构建、训练、评估与预测全流程，最终在验证集上达到 **97.80%** 的准确率。

**2. 核心模型与原理**  
- **模型名称**：MobileNetV3-Large（一种轻量级 CNN 架构）。  
- **核心思想**：**迁移学习**。  
  - 加载在 ImageNet（百万级图像）上预训练的 MobileNetV3 模型，作为特征提取器。  
  - **冻结**骨干网络的所有参数，仅替换最后的分类层（从 1000 类改为 2 类）。  
  - 使用少量猫狗数据训练新分类层，大幅降低训练成本，同时获得高精度。  
- **CNN 原理**：通过卷积层自动提取图像特征（边缘、纹理、形状），池化层降低维度，全连接层进行分类。

**3. 技术实现要点**  
- **框架**：PaddlePaddle（百度飞桨）。  
- **数据增强**：随机裁剪、水平翻转、颜色抖动，提升模型泛化能力。  
- **优化策略**：Adam 优化器 + 交叉熵损失，训练 10 个 epoch。  
- **环境适配**：解决 AI Studio 共享内存不足问题（`num_workers=0`），切换 GPU 加速。

**4. 成果与验证**  
- **验证集准确率**：97.80%（epoch 10 最佳模型）。  
- **单张图片预测**：可对新图片输出猫/狗类别及置信度（测试示例中置信度约 65%~73%）。  
- **可视化**：绘制训练/验证损失与准确率曲线，监控模型收敛情况。

**5. 项目意义**  
- 掌握了迁移学习在图像分类中的完整应用流程。  
- 熟悉了真实数据集（非标准目录结构）的处理方法。  
- 积累了云端深度学习环境（AI Studio）的调试经验。

**注**：预测置信度偏低可能与测试集图片预处理或模型过拟合有关，后续可通过微调（解冻部分骨干层）或扩大数据集进一步优化。

In [ ]:
# 第一步：数据解压与准备
import os
import zipfile
import paddle
import paddle.vision.transforms as T
from paddle.io import Dataset, DataLoader
from PIL import Image

# 1. 请务必替换成你复制的真实路径！
zip_path = 'data/cat-dog/test.zip'
extract_path = './data' # 解压到当前目录下的data文件夹

# 解压数据集
if not os.path.exists(extract_path):
    print(f"开始解压文件: {zip_path}")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"解压完成！文件已存放于: {extract_path}")
        # 打印解压后的一级目录内容，确认解压是否成功
        print("解压后目录内容:", os.listdir(extract_path))
    except FileNotFoundError:
        print(f"错误：找不到zip文件，请检查路径: {zip_path}")
        print("提示：请先在AI Studio的“数据”选项卡中复制正确的文件路径。")
        exit() # 终止运行
else:
    print(f"目录 {extract_path} 已存在，跳过解压步骤。")

# 2. 设置数据加载路径
# 解压后的目录结构通常是 ./data/train/cats 和 ./data/train/dogs，请根据实际情况调整
train_data_dir = os.path.join(extract_path, 'train')
# 如果没有train子目录，你可能需要手动创建或调整代码
if not os.path.exists(train_data_dir):
    print(f"警告：训练目录 {train_data_dir} 不存在，请检查解压后的文件夹结构。")
    # 可以在这里打印目录树辅助排查
    for root, dirs, files in os.walk(extract_path):
        level = root.replace(extract_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        # 只打印前5个文件，避免输出过长
        for f in files[:5]:
            print('{}{}'.format(subindent, f))
    exit()
else:
    print(f"找到训练数据目录: {train_data_dir}")

# 第二步：数据预处理与加载器

In [ ]:
import os

extract_path = './data'  # 你的解压目标路径

# 打印完整目录树，方便查看层级
def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * level
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for f in files[:5]:   # 每个文件夹最多显示5个文件
            print('{}{}'.format(subindent, f))
        if len(files) > 5:
            print('{}... 等 {} 个文件'.format(subindent, len(files)))

list_files(extract_path)

In [ ]:
import paddle
import paddle.vision.transforms as T
from paddle.io import Dataset, DataLoader
import os
from PIL import Image

# --- 超参数配置 ---
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

# --- 数据预处理 ---
train_transforms = T.Compose([
    T.RandomResizedCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 自定义数据集类（适配文件名前缀）---
class CatDogDataset(Dataset):
    def __init__(self, data_dir, transforms=None, is_train=True):
        super().__init__()
        self.transforms = transforms
        self.samples = []
        
        # 遍历 data_dir 下的所有图片文件
        for file_name in os.listdir(data_dir):
            if not file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            # 根据文件名前缀分配标签：cat -> 0，dog -> 1
            if file_name.startswith('cat'):
                label = 0
            elif file_name.startswith('dog'):
                label = 1
            else:
                continue  # 跳过不符合命名规则的文件
            self.samples.append((os.path.join(data_dir, file_name), label))
        
        print(f"从 {data_dir} 加载了 {len(self.samples)} 张图片（猫: {sum(1 for _, l in self.samples if l==0)}, 狗: {sum(1 for _, l in self.samples if l==1)}）")

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transforms:
            img = self.transforms(img)
        return img, label

    def __len__(self):
        return len(self.samples)

# --- 创建数据集实例 ---
train_dir = './data/train'   # 训练图片目录
full_dataset = CatDogDataset(train_dir, transforms=train_transforms)

# 划分训练集和验证集（80% 训练，20% 验证）
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = paddle.io.random_split(full_dataset, [train_size, val_size])

# 注意：验证集应使用无增强的预处理，但 random_split 返回的 Subset 仍然会调用原 dataset 的 transforms
# 这里简单起见，我们对验证集单独创建一个使用 eval_transforms 的 dataset
# 更规范的做法是创建两个独立 dataset，但为了快速验证，我们采用如下技巧：
val_dataset_no_aug = CatDogDataset(train_dir, transforms=eval_transforms)
# 只取验证集对应的索引子集
val_indices = val_dataset.indices
val_dataset = paddle.io.Subset(val_dataset_no_aug, val_indices)

# --- 创建数据加载器 ---
# --- 创建数据加载器（修正共享内存问题）---到了第四步回来修改--内存不够导致的！！！
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0,               # 降低进程数，2不行，原来是4
    use_shared_memory=False      # 禁用共享内存
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=0,               # 降低进程数
    use_shared_memory=False
)
print(f"训练集批次数: {len(train_loader)}, 验证集批次数: {len(val_loader)}")

# 第三步：构建 MobileNetV3 迁移学习模型

In [ ]:
# 检查分类器结构

In [ ]:
print("原始分类器结构：")
print(model.classifier)

In [ ]:
import paddle
from paddle.vision.models import mobilenet_v3_large

# ---------- 1. 加载预训练的 MobileNetV3-Large 模型 ----------
model = mobilenet_v3_large(pretrained=True)

# ---------- 2. 冻结主干网络的所有参数 ----------
for param in model.parameters():
    param.trainable = False

# ---------- 3. 替换分类头（输出层），适配猫狗二分类 ----------
# 根据上一步的诊断，第一个 Linear 层的输入特征数为 960
input_features = 960

model.classifier = paddle.nn.Sequential(
    paddle.nn.Linear(input_features, 128),
    paddle.nn.ReLU(),
    paddle.nn.Dropout(0.5),
    paddle.nn.Linear(128, 2)   # 二分类输出：猫(0) 和 狗(1)
)

# 打印模型结构，确认修改正确
paddle.summary(model, (1, 3, IMG_SIZE, IMG_SIZE))

# ---------- 4. 定义损失函数和优化器 ----------
criterion = paddle.nn.CrossEntropyLoss()
optimizer = paddle.optimizer.Adam(learning_rate=LEARNING_RATE, parameters=model.parameters())

print("模型构建完成！")

# 第四步：训练循环

In [ ]:
import time
import numpy as np
from tqdm import tqdm
from visualdl import LogWriter

# ---------- 初始化 VisualDL 记录器 ----------
log_writer = LogWriter(logdir="./log")

best_acc = 0.0

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # ---------- 训练阶段 ----------
    model.train()
    total_loss = 0.0
    correct_train = 0
    total_train = 0
    
    train_loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]')
    for batch_id, (images, labels) in enumerate(train_loop):
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        
        # 使用 .item() 提取标量值
        total_loss += loss.item()
        preds = paddle.argmax(outputs, axis=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.shape[0]
        
        train_loop.set_postfix(loss=loss.item())
    
    avg_train_loss = total_loss / len(train_loader)
    train_acc = correct_train / total_train
    
    log_writer.add_scalar(tag='train/loss', value=avg_train_loss, step=epoch)
    log_writer.add_scalar(tag='train/acc', value=train_acc, step=epoch)
    
    # ---------- 验证阶段 ----------
    model.eval()
    correct_val = 0
    total_val = 0
    val_loss = 0.0
    
    with paddle.no_grad():
        val_loop = tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Eval]')
        for batch_id, (images, labels) in enumerate(val_loop):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            preds = paddle.argmax(outputs, axis=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.shape[0]
    
    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct_val / total_val
    
    log_writer.add_scalar(tag='val/loss', value=avg_val_loss, step=epoch)
    log_writer.add_scalar(tag='val/acc', value=val_acc, step=epoch)
    
    # 保存最佳模型
    if val_acc > best_acc:
        best_acc = val_acc
        paddle.save(model.state_dict(), 'best_model.pdparams')
        print(f'  --> 新的最佳模型已保存，验证准确率: {best_acc:.4f}')
    
    epoch_time = time.time() - epoch_start
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Time: {epoch_time:.2f}s')

print(f'\n训练完成！最佳验证准确率: {best_acc:.4f}')

In [ ]:
# 想要读取数据可视化

In [ ]:
import matplotlib.pyplot as plt

# 从你的训练输出中提取的数据
epochs = list(range(1, 11))

train_loss = [0.2269, 0.1859, 0.1760, 0.1724, 0.1656, 0.1663, 0.1648, 0.1607, 0.1563, 0.1560]
train_acc  = [0.8992, 0.9219, 0.9244, 0.9264, 0.9292, 0.9262, 0.9303, 0.9313, 0.9308, 0.9355]

val_loss = [0.0773, 0.0700, 0.0670, 0.0641, 0.0619, 0.0642, 0.0631, 0.0611, 0.0679, 0.0609]
val_acc  = [0.9710, 0.9714, 0.9738, 0.9752, 0.9756, 0.9734, 0.9752, 0.9748, 0.9722, 0.9780]

# 创建图表
plt.figure(figsize=(14, 5))

# 左侧：损失曲线
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, 'b-o', label='Training Loss', linewidth=2, markersize=6)
plt.plot(epochs, val_loss, 'r-s', label='Validation Loss', linewidth=2, markersize=6)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# 右侧：准确率曲线
plt.subplot(1, 2, 2)
plt.plot(epochs, train_acc, 'b-o', label='Training Accuracy', linewidth=2, markersize=6)
plt.plot(epochs, val_acc, 'r-s', label='Validation Accuracy', linewidth=2, markersize=6)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Training and Validation Accuracy', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# 标注最佳验证准确率
best_epoch = val_acc.index(max(val_acc)) + 1
best_val = max(val_acc)
plt.annotate(f'Best: {best_val:.4f}', 
             xy=(best_epoch, best_val), 
             xytext=(best_epoch+0.5, best_val-0.005),
             arrowprops=dict(arrowstyle='->', color='green'),
             fontsize=10, color='green')

plt.tight_layout()
plt.show()

print(f"✅ 最佳验证准确率: {best_val:.4f} (出现在 Epoch {best_epoch})")

# 另外！！------训练完成后的标准流程

###  第一步：加载最佳模型并评估
之前的每当验证准确率提升，模型会自动保存为 best_model.pdparams

In [ ]:
import paddle
from paddle.vision.models import mobilenet_v3_large

# 加载最佳模型（注意 num_classes=2）
best_model = mobilenet_v3_large(pretrained=False, num_classes=2)
best_model.set_state_dict(paddle.load('best_model.pdparams'))
best_model.eval()

# 在验证集上计算准确率
correct = 0
total = 0
with paddle.no_grad():
    for images, labels in val_loader:
        outputs = best_model(images)
        preds = paddle.argmax(outputs, axis=1)
        correct += (preds == labels).sum().item()
        total += labels.shape[0]
print(f'✅ 最佳模型在验证集上的准确率: {correct / total:.4f}')

### 第二步：单张图片预测（体验模型效果）
找一张猫或狗的照片，或者从测试集中挑一张，测试模型的实际表现

In [ ]:
from PIL import Image
import paddle.vision.transforms as T

# 定义与验证集相同的预处理
eval_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_image(image_path, model, transforms):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transforms(img).unsqueeze(0)  # 增加 batch 维度
    
    with paddle.no_grad():
        output = model(img_tensor)
        pred_class = paddle.argmax(output, axis=1).item()
        confidence = paddle.nn.functional.softmax(output, axis=1)[0][pred_class].item()
    
    class_names = ['猫', '狗']
    print(f"图片路径: {image_path}")
    print(f"预测结果: {class_names[pred_class]}，置信度: {confidence:.4f}")

# 使用示例（请替换为实际图片路径）
predict_image('./data/test1/10001.jpg', best_model, eval_transforms)
predict_image('./data/test1/1.jpg', best_model, eval_transforms)
predict_image('./data/test1/10.jpg', best_model, eval_transforms)

### 第三步：批量预测测试集

In [ ]:
import os
import pandas as pd

test_dir = './data/test1'
test_files = [f for f in os.listdir(test_dir) if f.lower().endswith('.jpg')]

results = []
for file_name in test_files:
    img_path = os.path.join(test_dir, file_name)
    img = Image.open(img_path).convert('RGB')
    img_tensor = eval_transforms(img).unsqueeze(0)
    with paddle.no_grad():
        output = best_model(img_tensor)
        pred = paddle.argmax(output, axis=1).item()  # 0=猫，1=狗
    img_id = int(os.path.splitext(file_name)[0])    # 提取数字ID
    results.append({'id': img_id, 'label': pred})

# 按ID排序并保存
df = pd.DataFrame(results).sort_values('id')
df.to_csv('submission.csv', index=False)
print(f"✅ 预测完成，结果已保存至 submission.csv，共 {len(df)} 张图片。")

### 第四步：模型微调以进一步提升精度

原理：解冻预训练模型的最后几层，用很小的学习率继续训练，让模型更好地适应你的数据。

操作：加载最佳模型 → 解冻部分骨干层 → 降低学习率 → 再训练 5～10 轮。